# XAI-Guided Drift-Adaptive Streaming Fraud Detection
## Chapter 3 analysis notebook: data collection, EDA, cleansing, feature extraction, stream simulation, baselines and prequential fine-tuning

**Author:** Dinesh Gudapuri (Student ID C5054563), MSc Big Data Analytics, Sheffield Hallam University
**Supervisor:** Olamilekan Shobayo
**Notebook:** `Dinesh_Gudapuri_C5054563_Fraud_XAI.ipynb`

Dataset: ULB / Worldline *Credit Card Fraud Detection* (Dal Pozzolo et al.), Kaggle `mlg-ulb/creditcardfraud`, licence ODbL v1.0 (database) and DbCL v1.0 (contents). Mirror used for download: OpenML data id 1597. The raw data stays in `data/` and is **not** committed to GitHub (UREC2).

Rules from the UREC2 ethics form: one dataset; prequential (test-then-train) evaluation only, no cross-validation or random splits; metrics PR-AUC, G-mean and Precision@k (accuracy is not used); results as mean ± SD over ≥ 5 seeds.

| Section | Notebook part | Objective |
|---|---|---|
| 3.18 | Data collection, licence and storage | O1 |
| 3.19 | Exploratory data analysis | O1 |
| 3.20 | Data cleansing and validation rules | O1 |
| 3.21 | Feature extraction and engineering | O1, O4 |
| 3.22 | Stream simulation: delayed labels and drift injection | O2, O4 |
| 3.23 | Model selection and baselines | O1, O3, O4 |
| 3.24 | Prequential hyperparameter fine-tuning | O1, O5 |


In [ ]:
import sys, json, time, platform
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
import river, shap, sklearn
from fraudxai import config, data, cleansing, features, drift, metrics, stream, models, detectors, tuning

FIG = ROOT / "results" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
RES_FILE = ROOT / "results" / "results.json"
R = {}                                              # every number reported in Chapter 3
def save_results():
    RES_FILE.write_text(json.dumps(R, indent=2, default=float))

plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "Times New Roman"],
                     "font.size": 10, "axes.titlesize": 10, "figure.dpi": 110, "savefig.dpi": 300,
                     "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False})
C_LEGIT, C_FRAUD = "#2E7D32", "#C62828"
pd.set_option("display.width", 140, "display.max_columns", 20)
R["environment"] = {"python": platform.python_version(), "river": river.__version__, "shap": shap.__version__,
                    "scikit_learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__}
print(R["environment"])

## 3.18 Data collection, licence and storage
The parquet copy of OpenML 1597 is downloaded once into `data/` (git-ignored). Row and fraud counts are checked against the published figures (284,807 transactions, 492 frauds).

In [ ]:
path = data.download()
raw = data.load(path)
n, n_fraud = len(raw), int(raw.Class.sum())
R["dataset"] = {"source": "OpenML 1597 (mirror of Kaggle mlg-ulb/creditcardfraud)", "file_md5": data.md5(path),
                "rows": n, "columns": raw.shape[1], "fraud": n_fraud, "legit": n - n_fraud,
                "fraud_rate_pct": round(100 * n_fraud / n, 4),
                "time_span_hours": float(raw.Time.max() / 3600),
                "matches_published": (n == config.EXPECTED_ROWS) and (n_fraud == config.EXPECTED_FRAUD)}
save_results(); pd.Series(R["dataset"])

## 3.19 Exploratory data analysis
Descriptive statistics by class, class balance, Amount by class (log scale), transactions per hour and correlation with the label.

In [ ]:
desc_cols = ["Time", "Amount", "V17", "V14", "V12", "V10"]
desc = raw.groupby("Class")[desc_cols].describe().T.unstack(level=1)
tbl14 = raw[desc_cols].describe().T[["mean", "std", "min", "50%", "max"]]
tbl14["mean_legit"] = raw[raw.Class == 0][desc_cols].mean()
tbl14["mean_fraud"] = raw[raw.Class == 1][desc_cols].mean()
tbl14 = tbl14.round(2)
R["eda_descriptive"] = tbl14.to_dict(orient="index")
tbl14

In [ ]:
counts = raw.Class.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5.2, 3.0))
bars = ax.bar(["Legitimate (0)", "Fraud (1)"], counts.values, color=[C_LEGIT, C_FRAUD], width=0.55)
ax.set_yscale("log"); ax.set_ylabel("Transactions (log scale)")
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v * 1.15, f"{v:,}\n({100*v/n:.3f}%)", ha="center", fontsize=9)
ax.set_ylim(1, counts.max() * 8)
fig.savefig(FIG / "fig09_class_balance.png"); plt.show()
R["eda_class_balance"] = {"legit": int(counts[0]), "fraud": int(counts[1]),
                          "imbalance_ratio_legit_per_fraud": round(counts[0] / counts[1], 1)}

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.2))
amt = [raw.loc[raw.Class == c, "Amount"] + 1 for c in (0, 1)]
bp = ax.boxplot(amt, tick_labels=["Legitimate (0)", "Fraud (1)"], widths=0.5, patch_artist=True,
                flierprops=dict(marker=".", markersize=2, alpha=0.3))
for patch, col in zip(bp["boxes"], [C_LEGIT, C_FRAUD]):
    patch.set_facecolor(col); patch.set_alpha(0.55)
ax.set_yscale("log"); ax.set_ylabel("Amount + 1 (EUR, log scale)")
fig.savefig(FIG / "fig10_amount_by_class.png"); plt.show()
amt_stats = raw.groupby("Class").Amount.agg(["median", "mean", "max", lambda s: (s == 0).sum()])
amt_stats.columns = ["median", "mean", "max", "zero_amount_count"]
R["eda_amount_by_class"] = amt_stats.round(2).to_dict(orient="index")
amt_stats.round(2)

In [ ]:
hours = (raw.Time // 3600).astype(int)                # elapsed hour 0-47 (two days)
per_hour = raw.groupby([hours, "Class"]).size().unstack(fill_value=0)
rate = 100 * per_hour[1] / per_hour.sum(axis=1)
fig, ax1 = plt.subplots(figsize=(6.2, 3.2))
ax1.bar(per_hour.index, per_hour[0], color=C_LEGIT, alpha=0.6, label="Legitimate per hour")
ax1.set_xlabel("Elapsed hour since first transaction (0-47)"); ax1.set_ylabel("Legitimate transactions")
ax2 = ax1.twinx(); ax2.spines["right"].set_visible(True)
ax2.plot(per_hour.index, rate, color=C_FRAUD, marker="o", ms=2.5, lw=1.2, label="Fraud rate (%)")
ax2.set_ylabel("Fraud rate (%)")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper center", fontsize=8, frameon=False, ncol=2)
fig.savefig(FIG / "fig11_transactions_per_hour.png"); plt.show()
R["eda_per_hour"] = {"min_tx_per_hour": int(per_hour.sum(axis=1).min()), "max_tx_per_hour": int(per_hour.sum(axis=1).max()),
                     "max_fraud_rate_pct": round(float(rate.max()), 3), "hour_of_max_fraud_rate": int(rate.idxmax()),
                     "mean_fraud_rate_pct": round(float(rate.mean()), 3)}
R["eda_per_hour"]

In [ ]:
corr = raw.drop(columns="Time").corr()["Class"].drop("Class")
top = corr.abs().sort_values(ascending=False).head(10).index.tolist()
cm = raw[top + ["Class"]].corr()
fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(cm.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cm)), cm.columns, rotation=90, fontsize=8); ax.set_yticks(range(len(cm)), cm.columns, fontsize=8)
for i in range(len(cm)):
    for j in range(len(cm)):
        ax.text(j, i, f"{cm.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson r")
fig.savefig(FIG / "fig12_correlation_heatmap.png"); plt.show()
R["eda_corr_with_class_top10"] = corr[top].round(3).to_dict()
corr[top].round(3)

In [ ]:
save_results()

## 3.20 Data cleansing and validation rules
Audit every column, apply validation rules R1–R9, then clean (stable sort by Time, drop exact duplicates). Nothing is imputed or capped: large amounts are real and informative.

In [ ]:
audit = cleansing.audit_table(raw)
R["data_audit"] = {"columns": len(audit), "total_missing": int(audit.missing.sum()),
                   "dtypes": audit.dtype.value_counts().to_dict()}
audit.round(2).head(8)

In [ ]:
rules = pd.DataFrame([r.as_dict() for r in cleansing.validate(raw)])
df, clean_log = cleansing.clean(raw)
R["validation_rules"] = rules.to_dict(orient="records"); R["cleansing"] = clean_log
save_results()
print(rules.to_string(index=False)); print(clean_log)

## 3.21 Feature extraction and engineering
No card or customer ID exists, so only stream-level features are possible. All window features look back only (no future data). V1–V28 are already PCA components and are kept as they are.

In [ ]:
fe = features.add_features(df)
eng = fe[features.ENGINEERED + ["Class"]]
tbl17 = eng.groupby("Class")[features.ENGINEERED].mean().T.round(3)
tbl17.columns = ["mean_legit", "mean_fraud"]
R["features"] = {"model_features": features.MODEL_FEATURES, "n_model_features": len(features.MODEL_FEATURES),
                 "engineered_means_by_class": tbl17.to_dict(orient="index")}
# check that the online (one-at-a-time) builder gives the same values as the batch builder
online = features.OnlineFeatures(); sample = df.iloc[:5000].to_dict("records")
on = pd.DataFrame([online.transform_one(x) for x in sample])[features.ENGINEERED]
diff = float((on - fe.iloc[:5000][features.ENGINEERED].reset_index(drop=True)).abs().max().max())
R["features"]["online_vs_batch_max_abs_diff_first_5000"] = float(diff)
save_results(); print("max |online - batch| on first 5,000 rows:", diff); tbl17

## 3.22 Stream simulation: delayed labels and drift injection
The cleaned data is replayed in time order. The first 20 % is the warm-up slice (tuning only). Label delay is simulated in seconds of stream time; drift is injected after the warm-up slice, at the middle of the evaluation part.

In [ ]:
n_warm = int(len(fe) * config.WARMUP_FRACTION)
warm, evalp = fe.iloc[:n_warm].reset_index(drop=True), fe.iloc[n_warm:].reset_index(drop=True)
t0 = len(evalp) // 2
DELAYS = {"none (0 s)": 0, "1 hour": 3600, "6 hours": 21600}
# label backlog: number of earlier transactions whose label is still unknown (inside the delay window)
t_eval = evalp.Time.to_numpy(); idx = np.arange(len(t_eval))
backlog = {k: float(np.mean(idx - np.searchsorted(t_eval, t_eval - d, side="right"))) if d else 0.0
           for k, d in DELAYS.items()}
R["stream"] = {"warmup_rows": n_warm, "warmup_fraud": int(warm.Class.sum()), "warmup_end_hour": float(warm.Time.iloc[-1] / 3600),
               "eval_rows": len(evalp), "eval_fraud": int(evalp.Class.sum()), "drift_t0_in_eval": int(t0),
               "drift_t0_hour": float(evalp.Time.iloc[t0] / 3600), "delays_s": DELAYS,
               "mean_unlabelled_backlog": backlog,
               "drift_pool_matches_warmup_top8": set(drift.TOP_FEATURES) == set(warm.drop(columns="Time")[config.V_COLS + ["Class"]].corr()["Class"].drop("Class").abs().nlargest(8).index)}
pd.Series(R["stream"])

In [ ]:
scen = []
for kind in ["abrupt", "gradual", "adversarial"]:
    _, info = drift.inject(evalp, kind, t0=t0, seed=0)
    scen.append({"scenario": kind, **{k: v for k, v in info.items() if k not in ("kind", "seed")}})
warm_top8 = warm.drop(columns="Time")[config.V_COLS + ["Class"]].corr()["Class"].drop("Class").abs().nlargest(8).index.tolist()
R["drift_scenarios_seed0"] = scen; R["drift_feature_pool_from_warmup"] = warm_top8
save_results(); print("top-8 |corr| features in warm-up:", warm_top8); pd.DataFrame(scen)